<a href="https://colab.research.google.com/github/yanhanchen110/Sudoku-AI-Analysis/blob/main/Colab%E8%82%A1%E7%A5%A8%E6%8A%95%E8%B3%87%E5%88%86%E6%9E%90APP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# ==========================================================================
# 擴充步驟：加入「台灣核心權值股」自動掃描推薦功能
# ==========================================================================
import time

# 定義台灣市值前 50 大與核心熱門權值股清單 (過濾掉流動性極差或不適用的標的)
TAIWAN_CORE_STOCKS = [
    '2330.TW', '2317.TW', '2454.TW', '2308.TW', '2881.TW', '2882.TW', '2382.TW',
    '2891.TW', '3008.TW', '2303.TW', '3711.TW', '2412.TW', '2886.TW', '2884.TW',
    '2892.TW', '2885.TW', '2327.TW', '2880.TW', '1301.TW', '1303.TW', '1216.TW',
    '2002.TW', '2357.TW', '2395.TW', '3231.TW', '2345.TW', '6505.TW', '2883.TW',
    '2887.TW', '2890.TW', '5880.TW', '1101.TW', '2603.TW', '2609.TW', '2615.TW',
    '2301.TW', '2379.TW', '3037.TW', '3045.TW', '2408.TW', '2409.TW', '3481.TW',
    '2801.TW', '2834.TW', '6669.TW', '2356.TW', '2324.TW', '2474.TW', '2353.TW'
]

# 建立第三個按鈕：掃描黑馬
btn_scan_all = widgets.Button(
    description='🔥 掃描台股權值黑馬',
    button_style='warning', # 黃色按鈕
    tooltip='自動掃描台灣核心權值股，篩選出目前 4~5 星的即時推薦名單',
    icon='rocket'
)

# 點擊「掃描台股權值黑馬」的事件處理
def on_scan_all_clicked(b):
    with output_area:
        clear_output()
        print("🚀 開始自動掃描台灣核心權值股 (約 50 檔)...")
        print("⏱️ 為了避免被 Yahoo 鎖 IP，每檔股票將進行快速計算，預計需要 30 ~ 45 秒，請稍候...\n")

        recommend_list = []
        progress_bar = widgets.IntProgress(value=0, min=0, max=len(TAIWAN_CORE_STOCKS), description='進度:', style={'description_width': 'initial'})
        display(progress_bar)

        for ticker in TAIWAN_CORE_STOCKS:
            try:
                # 1. 抓取過去 365 天歷史資料計算現狀
                end_date = datetime.datetime.now().strftime('%Y-%m-%d')
                start_date = (datetime.datetime.now() - datetime.timedelta(days=365)).strftime('%Y-%m-%d')
                df_daily = yf.download(ticker, start=start_date, end=end_date, progress=False)

                if df_daily.empty:
                    progress_bar.value += 1
                    continue
                if isinstance(df_daily.columns, pd.MultiIndex):
                    df_daily.columns = df_daily.columns.get_level_values(0)

                df_daily.index = pd.to_datetime(df_daily.index)
                df_weekly = df_daily.resample('W').last()

                close_weekly = df_weekly['Close'].squeeze().astype(float)
                df_weekly['MA20'] = ta.trend.sma_indicator(close_weekly, window=20)
                stoch = ta.momentum.StochasticOscillator(high=df_weekly['High'].squeeze().astype(float), low=df_weekly['Low'].squeeze().astype(float), close=close_weekly, window=9, smooth_window=3)
                df_weekly['K'] = stoch.stoch()
                df_weekly['D'] = stoch.stoch_signal()

                close_daily = df_daily['Close'].squeeze().astype(float)
                indicator_bb = ta.volatility.BollingerBands(close=close_daily, window=20, window_dev=2)
                df_daily['BB_high'] = indicator_bb.bollinger_hband()
                df_daily['BB_low'] = indicator_bb.bollinger_lband()
                df_daily['BB_mid'] = indicator_bb.bollinger_mavg()

                last_daily = df_daily.iloc[-1]
                last_weekly = df_weekly.iloc[-1]
                prev_weekly = df_weekly.iloc[-2]
                current_price = float(last_daily['Close'])

                is_weekly_bullish = current_price > float(last_weekly['MA20'])
                weekly_gold_cross = (float(prev_weekly['K']) <= float(prev_weekly['D'])) and (float(last_weekly['K']) > float(last_weekly['D']))
                is_daily_oversold = current_price <= float(last_daily['BB_low']) * 1.02

                # 計算基礎型態分數 (掃描時採快速型態打分，避免頻繁調用 5 年回測導致被 Yahoo 鎖 IP)
                score = 30
                if is_weekly_bullish: score += 30
                if weekly_gold_cross: score += 25
                if is_daily_oversold: score += 15

                stars = max(1, min(5, round(score / 20)))

                # 我們只挑選出目前處於 4 星 與 5 星 的績優波段黑馬
                if stars >= 4:
                    if stars == 5:
                        comment = "🟢 強力推薦：長線多頭且短線剛好回檔至布林下軌超賣區，時機非常完美！"
                    else:
                        comment = "🔵 值得投資：整體多頭架構完好，短線動能蓄勢待發，建議逢低分批布局。"

                    recommend_list.append({
                        '代號': ticker.replace('.TW', ''),
                        '當前價': f"${current_price:.2f}",
                        '評級': "🔥" * stars,
                        '操盤手大師短評': comment,
                        'stars_num': stars
                    })
            except:
                pass

            progress_bar.value += 1
            time.sleep(0.1) # 稍微冷卻避免被阻斷

        clear_output()

        # 顯示篩選結果
        if not recommend_list:
            display(Markdown("### 📭 掃描完成：今日台股大盤權值股中，暫時沒有符合 4~5 星的拉回起漲黑馬，建議空倉觀望。"))
        else:
            # 依照星等由高到低排序
            recommend_list = sorted(recommend_list, key=lambda x: x['stars_num'], reverse=True)

            md_result = "## 🎯 今日 AI 智慧選股・台股核心權值推薦名單\n"
            md_result += "> 系統已自動幫你過濾掉趨勢走空、高檔過熱的股票，以下為目前最符合「週線多頭＋日線撿便宜」的標的：\n\n"
            md_result += "| 股票代號 | 當前價格 | AI 綜合評等 | 操盤手大師短評 |\n"
            md_result += "| :--- | :--- | :--- | :--- |\n"

            for stock in recommend_list:
                md_result += f"| **{stock['代號']}** | {stock['當前價']} | {stock['評級']} | {stock['操盤手大師短評']} |\n"

            md_result += "\n💡 *提示：你可以複製上表中的代號，輸入到左側文字框中點擊「生成當前分析」，即可查閱該股的詳細安全買進區間與停損停利點！*"
            display(Markdown(md_result))

# 綁定新按鈕事件
btn_scan_all.on_click(on_scan_all_clicked)

# 清除舊的介面顯示，重新把三個按鈕排在一起
clear_output()
print("==========================================================================================")
print(" 👑 股票週線波段分析 ＋ 歷史勝率回測 ＋ AI 智慧推薦星等 ＋ 權值股即時掃描選股系統 (升級完成) ")
print("==========================================================================================")
display(widgets.HBox([input_ticker, btn_analyze, btn_backtest, btn_scan_all]))
display(output_area)

 👑 股票週線波段分析 ＋ 歷史勝率回測 ＋ AI 智慧推薦星等 ＋ 權值股即時掃描選股系統 (升級完成) 


Output()

In [8]:
# ==========================================
# 步驟 1：安裝並匯入所有必要的 Python 套件
# ==========================================
!pip install yfinance ta ipywidgets -q

import datetime
import pandas as pd
import yfinance as yf
import ta
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# ==========================================
# 輔助函式：為了計算推薦星等，內部默默執行精簡版回測
# ==========================================
def _get_backtest_score(ticker_symbol):
    """供分析報告調用，快速計算歷史勝率與報酬率得分"""
    end_date = datetime.datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.datetime.now() - datetime.timedelta(days=5*365)).strftime('%Y-%m-%d')
    df = yf.download(ticker_symbol, start=start_date, end=end_date, progress=False)
    if df.empty or len(df) < 40:
        return 0, 0, 0
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    close_prices = df['Close'].squeeze().astype(float)
    high_prices = df['High'].squeeze().astype(float)
    low_prices = df['Low'].squeeze().astype(float)

    indicator_bb = ta.volatility.BollingerBands(close=close_prices, window=20, window_dev=2)
    df['BB_high'] = indicator_bb.bollinger_hband()
    df['BB_mid'] = indicator_bb.bollinger_mavg()
    df['BB_low'] = indicator_bb.bollinger_lband()

    stoch = ta.momentum.StochasticOscillator(high=high_prices, low=low_prices, close=close_prices, window=9, smooth_window=3)
    df['K'] = stoch.stoch()
    df['D'] = stoch.stoch_signal()

    in_position = False
    buy_price = 0
    trades = []

    for i in range(20, len(df)):
        current_close = float(df['Close'].iloc[i])
        current_bb_low = float(df['BB_low'].iloc[i])
        current_bb_high = float(df['BB_high'].iloc[i])
        k_today, d_today = float(df['K'].iloc[i]), float(df['D'].iloc[i])
        k_yesterday, d_yesterday = float(df['K'].iloc[i-1]), float(df['D'].iloc[i-1])

        gold_cross = (k_yesterday <= d_yesterday) and (k_today > d_today)
        is_oversold = current_close <= current_bb_low * 1.02

        if not in_position:
            if is_oversold or (gold_cross and current_close > float(df['BB_mid'].iloc[i])):
                in_position = True
                buy_price = current_close
        else:
            if current_close >= current_bb_high or current_close <= buy_price * 0.93:
                in_position = False
                trades.append((current_close - buy_price) / buy_price * 100)

    if not trades:
        return 0, 0, 0

    win_rate = (len([p for p in trades if p > 0]) / len(trades)) * 100
    total_return = sum(trades)

    first_close = float(df['Close'].iloc[20])
    last_close = float(df['Close'].iloc[-1])
    bh_return = (last_close - first_close) / first_close * 100

    # 回測表現相較於買入持有的超額收益
    outperform = total_return - bh_return
    return win_rate, total_return, outperform

# ==========================================
# 步驟 2：核心分析與智慧推薦生成邏輯
# ==========================================
def analyze_stock(ticker_symbol):
    end_date = datetime.datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.datetime.now() - datetime.timedelta(days=365)).strftime('%Y-%m-%d')

    df_daily = yf.download(ticker_symbol, start=start_date, end=end_date, progress=False)
    if df_daily.empty:
        return f"❌ 找不到股票代號 【{ticker_symbol}】 的資料，請檢查代號是否正確。"

    if isinstance(df_daily.columns, pd.MultiIndex):
        df_daily.columns = df_daily.columns.get_level_values(0)

    df_daily.index = pd.to_datetime(df_daily.index)
    df_weekly = df_daily.resample('W').last()

    close_weekly = df_weekly['Close'].squeeze().astype(float)
    high_weekly = df_weekly['High'].squeeze().astype(float)
    low_weekly = df_weekly['Low'].squeeze().astype(float)

    df_weekly['MA20'] = ta.trend.sma_indicator(close_weekly, window=20)
    stoch = ta.momentum.StochasticOscillator(high=high_weekly, low=low_weekly, close=close_weekly, window=9, smooth_window=3)
    df_weekly['K'] = stoch.stoch()
    df_weekly['D'] = stoch.stoch_signal()

    close_daily = df_daily['Close'].squeeze().astype(float)
    indicator_bb = ta.volatility.BollingerBands(close=close_daily, window=20, window_dev=2)
    df_daily['BB_high'] = indicator_bb.bollinger_hband()
    df_daily['BB_low'] = indicator_bb.bollinger_lband()
    df_daily['BB_mid'] = indicator_bb.bollinger_mavg()

    last_daily = df_daily.iloc[-1]
    last_weekly = df_weekly.iloc[-1]
    prev_weekly = df_weekly.iloc[-2]
    current_price = float(last_daily['Close'])

    # 判斷當前指標狀態
    is_weekly_bullish = current_price > float(last_weekly['MA20'])
    weekly_gold_cross = (float(prev_weekly['K']) <= float(prev_weekly['D'])) and (float(last_weekly['K']) > float(last_weekly['D']))
    is_daily_oversold = current_price <= float(last_daily['BB_low']) * 1.02

    # 👑 【核心新增】智慧評級分數計算 (總分 100)
    score = 20 # 基礎分
    if is_weekly_bullish: score += 25       # 多頭趨勢加分
    if weekly_gold_cross: score += 20       # 黃金交叉加分
    if is_daily_oversold: score += 15       # 回檔超賣區加分

    # 撈取歷史回測表現來幫現在的推薦度「背書」
    win_rate, _, outperform = _get_backtest_score(ticker_symbol)
    if win_rate >= 60: score += 10          # 歷史勝率高加分
    if outperform > 0: score += 10          # 策略能打敗波段持有加分

    # 將分數換算成 1~5 顆星
    stars = max(1, min(5, round(score / 20)))
    star_display = "🔥" * stars + "✨" * (5 - stars)

    # 根據星等給予操盤手大師短評
    if stars == 5:
        recommend_text = "【強力推薦・時機完美】目前長線多頭確立、短線剛好回檔至黃金買點，且歷史勝率極高，屬於千載難逢的送分題型態！"
    elif stars == 4:
        recommend_text = "【值得投資・分批進場】整體架構良好，策略期望值高。可能需要一點耐心等待短線震盪噴發，適合穩健型買盤。"
    elif stars == 3:
        recommend_text = "【中性看待・謹慎試單】技術面雖有撐，但歷史數據顯示該股洗盤較為劇烈。若要參與，建議严格控制初始資金。"
    elif stars == 2:
        recommend_text = "【風險偏高・不入虎穴】目前不論是長線均線還是短線動能皆不明朗，進場容易上下挨巴掌，建議改看其他標的。"
    else:
        recommend_text = "【極度危險・空頭肆虐】指標全面走弱且跌破關鍵防守，歷史回測也顯示目前並非有利時機，請立刻移出觀察清單！"

    safe_buy_low = round(float(last_daily['BB_low']), 2)
    safe_buy_high = round(float(last_daily['BB_mid']), 2)
    target_price = round(float(last_daily['BB_high']), 2)

    # 生成包含「綜合推薦」的全新 Markdown 報告
    report = f"## 📊 【{ticker_symbol}】投資分析與決策簡報\n"
    report += f"**當前收盤價**：`${current_price:.2f}`\n\n"

    report += "---"
    report += f"### 👑 系統綜合投資推薦度\n"
    report += f"### **評級： {star_display} ({stars} / 5 星)**\n"
    report += f"> 💡 **大師短評**：{recommend_text}\n"
    report += "---\n\n"

    report += "### 🔍 技術面現狀摘要\n"
    report += f"- **20周均線波段型態**：${float(last_weekly['MA20']):.2f} (目前：{'🟢 多頭格局' if is_weekly_bullish else '🔴 空頭格局'})\n"
    report += f"- **最新周線KD指標**：`K={float(last_weekly['K']):.1f}`, `D={float(last_weekly['D']):.1f}`\n\n"

    report += "### 🎯 行動導向投資結論\n"
    if is_weekly_bullish and weekly_gold_cross:
        report += "➡️ **【這週可投資】** 周線級別黃金交叉，股價在趨勢線之上，強烈波段起漲訊號。\n"
    elif is_weekly_bullish and is_daily_oversold:
        report += "➡️ **【今日可投資】** 長線偏多但今日短線回檔至布林下軌超賣區，高勝率的拉回買點。\n"
    elif is_weekly_bullish and not weekly_gold_cross:
        report += "➡️ **【這個月可投資 (分批佈局)】** 中長線看好，但短線動能高檔震盪，建議本月逢低慢買。\n"
    else:
        report += "➡️ **【當前觀望 / 暫不建議投資】** 跌破長線均線或技術指標修正中，建議空倉等待止跌。\n"

    report += f"\n📋 **操作導航區間**：\n"
    report += f"- **🎯 安全買進區間**：`${safe_buy_low}` ~ `${safe_buy_high}`\n"
    report += f"- **🚀 波段停利目標**：`${target_price}`\n"
    report += f"- **⚠️ 建議停損價位**：`${round(safe_buy_low * 0.95, 2)}` (安全下限跌破 5%)\n"

    return report

# ==========================================
# 步驟 3：歷史策略回測邏輯
# ==========================================
def backtest_strategy(ticker_symbol):
    end_date = datetime.datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.datetime.now() - datetime.timedelta(days=5*365)).strftime('%Y-%m-%d')
    df = yf.download(ticker_symbol, start=start_date, end=end_date, progress=False)

    if df.empty: return "❌ 無法取得回測歷史資料。", None
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    df.index = pd.to_datetime(df.index)

    close_prices = df['Close'].squeeze().astype(float)
    high_prices = df['High'].squeeze().astype(float)
    low_prices = df['Low'].squeeze().astype(float)

    indicator_bb = ta.volatility.BollingerBands(close=close_prices, window=20, window_dev=2)
    df['BB_high'], df['BB_mid'], df['BB_low'] = indicator_bb.bollinger_hband(), indicator_bb.bollinger_mavg(), indicator_bb.bollinger_lband()

    stoch = ta.momentum.StochasticOscillator(high=high_prices, low=low_prices, close=close_prices, window=9, smooth_window=3)
    df['K'], df['D'] = stoch.stoch(), stoch.stoch_signal()

    in_position = False
    buy_price = 0
    trades = []

    for i in range(20, len(df)):
        current_date = df.index[i]
        current_close = float(df['Close'].iloc[i])
        current_bb_low = float(df['BB_low'].iloc[i])
        current_bb_high = float(df['BB_high'].iloc[i])
        k_today, d_today = float(df['K'].iloc[i]), float(df['D'].iloc[i])
        k_yesterday, d_yesterday = float(df['K'].iloc[i-1]), float(df['D'].iloc[i-1])

        gold_cross = (k_yesterday <= d_yesterday) and (k_today > d_today)
        is_oversold = current_close <= current_bb_low * 1.02

        if not in_position:
            if is_oversold or (gold_cross and current_close > float(df['BB_mid'].iloc[i])):
                in_position = True
                buy_price = current_close
                buy_date = current_date
        else:
            stop_loss = buy_price * 0.93
            if current_close >= current_bb_high or current_close <= stop_loss:
                in_position = False
                sell_price = current_close
                sell_date = current_date
                profit_pct = (sell_price - buy_price) / buy_price * 100
                trades.append({'buy_date': buy_date, 'sell_date': sell_date, 'buy_price': buy_price, 'sell_price': sell_price, 'profit_pct': profit_pct})

    if not trades: return "### 📊 歷史回測報告\n> 過去 5 年內此策略未觸發任何訊號。", df

    df_trades = pd.DataFrame(trades)
    total_trades = len(df_trades)
    winning_trades = len(df_trades[df_trades['profit_pct'] > 0])
    win_rate = (winning_trades / total_trades) * 100
    total_return = df_trades['profit_pct'].sum()
    avg_return = df_trades['profit_pct'].mean()

    first_close, last_close = float(df['Close'].iloc[20]), float(df['Close'].iloc[-1])
    bh_return = (last_close - first_close) / first_close * 100

    bt_report = f"## 📈 過去 5 年策略歷史回測數據統計 (標的: {ticker_symbol})\n"
    bt_report += f"- **總模擬交易次數**：`{total_trades}` 次\n"
    bt_report += f"- **策略勝率**：`{win_rate:.1f}%` (獲利：{winning_trades} 次 / 虧損：{total_trades - winning_trades} 次)\n"
    bt_report += f"- **策略大盤對比**：累計報酬 `{total_return:.2f}%` ｜ 對比不操作買入持有 `{bh_return:.2f}%` ｜ 策略超額損益 `{total_return - bh_return:.2f}%` \n\n"
    bt_report += "### 📝 最近 5 筆歷史模擬交易明細：\n"
    for _, row in df_trades.tail(5).iterrows():
        emoji = "🟢 獲利" if row['profit_pct'] > 0 else "🔴 虧損"
        bt_report += f"- {emoji} | 進場: {row['buy_date'].strftime('%Y-%m-%d')} (`${row['buy_price']:.2f}`) ➡️ 出場: {row['sell_date'].strftime('%Y-%m-%d')} (`${row['sell_price']:.2f}`) | **損益: {row['profit_pct']:.2f}%**\n"

    return bt_report, df

# ==========================================
# 步驟 4：建立互動 APP UI 介面
# ==========================================
input_ticker = widgets.Text(value='2330.TW', placeholder='例如: 2330.TW 或 AAPL', description='股票代號:')
btn_analyze = widgets.Button(description='生成當前分析', button_style='success', tooltip='分析目前的技術面型態與評星', icon='bar-chart')
btn_backtest = widgets.Button(description='執行 5年歷史回測', button_style='info', tooltip='驗證此策略過去5年歷史勝率', icon='history')
output_area = widgets.Output()

def on_analyze_clicked(b):
    with output_area:
        clear_output()
        ticker = input_ticker.value.strip().upper()
        print(f"🔍 正在解析 {ticker} 的技術數據，並根據回測結果計算 AI 推薦星等，請稍候...")
        try:
            report_md = analyze_stock(ticker)
            clear_output()
            display(Markdown(report_md))
        except Exception as e:
            clear_output()
            print(f"❌ 分析失敗，錯誤原因: {e}")

def on_backtest_clicked(b):
    with output_area:
        clear_output()
        ticker = input_ticker.value.strip().upper()
        print(f"⚙️ 正在追蹤 {ticker} 的歷史回測流水線...")
        try:
            backtest_md, _ = backtest_strategy(ticker)
            clear_output()
            display(Markdown(backtest_md))
        except Exception as e:
            clear_output()
            print(f"❌ 回測失敗: {e}")

btn_analyze.on_click(on_analyze_clicked)
btn_backtest.on_click(on_backtest_clicked)

print("==========================================================================")
print(" 👑 股票週線波段分析評估 ＋ 歷史勝率回測 ＋ AI 智慧推薦星等系統 (完全體) ")
print("==========================================================================")
display(widgets.HBox([input_ticker, btn_analyze, btn_backtest]))
display(output_area)

 👑 股票週線波段分析評估 ＋ 歷史勝率回測 ＋ AI 智慧推薦星等系統 (完全體) 


Output()